## Install Dependencies

In [ ]:
!pip install transformers torch pandas scikit-learn scipy openpyxl

## Mount Google Drive and Directories

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/DSP/'
MODEL_DIR = '/content/drive/MyDrive/DSP/dsp_toxicity_model/'

## Imports

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader, RandomSampler, SequentialSampler
from sklearn.model_selection import train_test_split
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_absolute_error
import re

## Load Data

In [ ]:
df = pd.read_excel(DATA_DIR + 'satc_full_Chris_2026_03_26.xlsx')
df = df.rename(columns={' ': 'category'})
df = df[df['Political Toxicity'].notna()].reset_index(drop=True)
print(f'Rated tweets: {len(df)}')

## Clean @mentions

In [ ]:
def normalize_mentions(text):
    return re.sub(r'@\S+', '@user', str(text))

df['text_clean'] = df['text_revised'].apply(normalize_mentions)

## Sample Weights

In [ ]:
df['Categoric Toxicity'] = pd.to_numeric(df['Categoric Toxicity'], errors='coerce')
df['Confidence'] = pd.to_numeric(df['Confidence'], errors='coerce')
df['weight'] = df['Categoric Toxicity'] * df['Confidence']
df['weight'] = df['weight'].fillna(1.0)
df['weight'] = df['weight'].replace(0, 0.1)
df['weight'] = df['weight'] / df['weight'].mean()
print("Any NaN in weights:", df['weight'].isna().any())
print("Min weight:", df['weight'].min())

## Holdout Split

In [ ]:
df['score_bin'] = pd.cut(df['Political Toxicity'], bins=[0,3,6,10], labels=['low','mid','high'], include_lowest=True)
df['stratum'] = df['category'] + '_' + df['PARTY'].fillna('Unknown') + '_' + df['score_bin'].astype(str)

train_val_df, holdout_df = train_test_split(
    df, test_size=0.15, random_state=42, stratify=df['stratum']
)
print(f'Train+val: {len(train_val_df)}, Holdout: {len(holdout_df)}')

## Tokenizer and Dataset Class

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

tokenizer = AutoTokenizer.from_pretrained('cardiffnlp/twitter-roberta-base-2022-154m')

class TweetDataset(Dataset):
    def __init__(self, texts, labels, weights, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.weights = weights
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            self.texts[idx],
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoded['input_ids'].squeeze(),
            'attention_mask': encoded['attention_mask'].squeeze(),
            'label': torch.tensor(self.labels[idx], dtype=torch.float),
            'weight': torch.tensor(self.weights[idx], dtype=torch.float)
        }

## DataLoaders

In [ ]:
train_df, val_df = train_test_split(train_val_df, test_size=0.1, random_state=42)

def make_loader(df, shuffle=True, batch_size=16):
    dataset = TweetDataset(
        df['text_clean'].tolist(),
        df['Political Toxicity'].tolist(),
        df['weight'].tolist(),
        tokenizer
    )
    sampler = RandomSampler(dataset) if shuffle else SequentialSampler(dataset)
    return DataLoader(dataset, sampler=sampler, batch_size=batch_size)

train_loader = make_loader(train_df, shuffle=True)
val_loader = make_loader(val_df, shuffle=False)
holdout_loader = make_loader(holdout_df, shuffle=False)

## Model

In [ ]:
class ToxicityRegressor(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = AutoModel.from_pretrained('cardiffnlp/twitter-roberta-base-2022-154m')
        self.regressor = nn.Linear(768, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_vector = outputs.last_hidden_state[:, 0, :]
        return self.regressor(cls_vector).squeeze()

model = ToxicityRegressor().to(device)

## Optimizer and Scheduler

In [ ]:
epochs = 2
optimizer = AdamW(model.parameters(), lr=2e-5, eps=1e-8)
total_steps = len(train_loader) * epochs
warmup_steps = int(0.1 * total_steps)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

## Training Loop

In [ ]:
train_losses = []
val_maes = []

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        weights = batch['weight'].to(device)

        model.zero_grad()
        preds = model(input_ids, attention_mask)
        loss = (weights * (preds - labels) ** 2).mean()

        if torch.isnan(loss):
            print(f'NaN loss detected')
            break

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    model.eval()
    val_preds, val_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            preds = model(input_ids, attention_mask)
            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(batch['label'].numpy())

    val_mae = mean_absolute_error(val_labels, val_preds)
    val_maes.append(val_mae)
    print(f'Epoch {epoch+1:>1} | Train Loss: {avg_train_loss:>8.4f} | Val MAE: {val_mae:>6.4f}')

## Training Curve

In [ ]:
import matplotlib.pyplot as plt

epochs_range = list(range(1, epochs + 1))

fig, ax1 = plt.subplots(figsize=(8, 4))

ax1.plot(epochs_range, train_losses, 'b-o', label='Train Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Train Loss', color='blue')
ax1.tick_params(axis='y', labelcolor='blue')
ax1.set_ylim(0, max(train_losses) * 1.1)

ax2 = ax1.twinx()
ax2.plot(epochs_range, val_maes, 'r-o', label='Val MAE')
ax2.set_ylabel('Val MAE', color='red')
ax2.tick_params(axis='y', labelcolor='red')
ax2.set_ylim(min(val_maes) * 0.95, max(val_maes) * 1.05)

fig.suptitle('Training Loss and Validation MAE by Epoch')
fig.legend(loc='upper right', bbox_to_anchor=(0.88, 0.88))
plt.tight_layout()
plt.show()

## Holdout Evaluation

In [ ]:
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for batch in holdout_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        preds = model(input_ids, attention_mask)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch['label'].numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

mae = mean_absolute_error(all_labels, all_preds)
pearson, _ = pearsonr(all_labels, all_preds)
spearman, _ = spearmanr(all_labels, all_preds)
mask = all_labels >= 7
mae_high = mean_absolute_error(all_labels[mask], all_preds[mask])

print(f'MAE:          {mae:.3f}')
print(f'Pearson r:    {pearson:.3f}')
print(f'Spearman rho: {spearman:.3f}')
print(f'MAE (>=7):    {mae_high:.3f}')

## Inter-rater Agreement

In [ ]:
layla = pd.read_excel(DATA_DIR + 'satc_full_layla.xlsx')
layla = layla.rename(columns={'condition': 'category'})

merged = pd.merge(
    df[['tweet_id_satc', 'Political Toxicity']].rename(columns={'Political Toxicity': 'chris_score'}),
    layla[['tweet_id_satc', 'Political Toxicity']].rename(columns={'Political Toxicity': 'layla_score'}),
    on='tweet_id_satc'
)

merged = merged.dropna(subset=['chris_score', 'layla_score'])
print(f'Overlapping rated tweets: {len(merged)}')

ira_mae = mean_absolute_error(merged['chris_score'], merged['layla_score'])
ira_pearson, _ = pearsonr(merged['chris_score'], merged['layla_score'])
ira_spearman, _ = spearmanr(merged['chris_score'], merged['layla_score'])

print(f'Inter-rater MAE:          {ira_mae:.3f}')
print(f'Inter-rater Pearson r:    {ira_pearson:.3f}')
print(f'Inter-rater Spearman rho: {ira_spearman:.3f}')

## Model vs. Human Inter-Rater Agreement

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

metrics = ['MAE', 'Pearson r', 'Spearman rho']
model_scores = [mae, pearson, spearman]
human_scores = [ira_mae, ira_pearson, ira_spearman]

x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 4))
bars1 = ax.bar(x - width/2, model_scores, width, label='Model', color='#4C72B0')
bars2 = ax.bar(x + width/2, human_scores, width, label='Human (Chris vs Layla)', color='#DD8452')

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=10)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=10)

ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_title('Model vs Human Inter-Rater Agreement')
ax.set_ylim(0, max(human_scores + model_scores) + 0.3)
ax.legend()
plt.tight_layout()
plt.show()

## Example Tweets

In [ ]:
df['text_length'] = df['text_revised'].apply(lambda x: len(str(x).split()))
short_df = df[df['text_length'] <= 30]

text_lookup = short_df.set_index('tweet_id_satc')['text_revised']
eligible = merged[merged['tweet_id_satc'].isin(short_df['tweet_id_satc'])]
sample = eligible.sample(5, random_state=42).reset_index(drop=True)
sample_texts = sample['tweet_id_satc'].map(text_lookup).tolist()

model.eval()
sample_preds = []
with torch.no_grad():
    for text in sample_texts:
        encoded = tokenizer(
            text,
            add_special_tokens=True,
            max_length=128,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        input_ids = encoded['input_ids'].to(device)
        attention_mask = encoded['attention_mask'].to(device)
        pred = model(input_ids, attention_mask)
        sample_preds.append(round(pred.item(), 2))

sample['model_score'] = sample_preds
sample['text'] = sample_texts

print(sample[['chris_score', 'layla_score', 'model_score', 'text']].to_string(index=False))

## Save Model

In [ ]:
model.bert.save_pretrained(MODEL_DIR)
torch.save(model.regressor.state_dict(), MODEL_DIR + 'regressor_head.pt')
tokenizer.save_pretrained(MODEL_DIR)
print('Model saved.')

##Results
#### First-Cut Model Results — DSP Political Toxicity Classifier

- Trained on 6,769 tweets rated by Chris across three categories: Media, Election, and Violence
- Base model: `cardiffnlp/twitter-roberta-base-2022-154m`, fine-tuned with a regression head to predict a continuous 0–10 political toxicity score
- 15% stratified holdout set reserved before training and used only for final evaluation

#### Holdout Performance

- MAE of 1.60 — the model is on average 1.60 points off from Chris's rating on unseen tweets
- Pearson r of 0.62 and Spearman rho of 0.61 — the model meaningfully tracks Chris's ranking of tweets by toxicity
- MAE on tweets rated ≥7 is 1.43 — the model performs slightly better at the high end of the scale

#### Human Baseline (Chris vs. Layla, 769 overlapping tweets)

- Inter-rater MAE of 2.30
- Inter-rater Pearson r of 0.44
- Inter-rater Spearman rho of 0.46

#### Interpretation

- The model outperforms human inter-rater agreement on all three metrics
- This advantage is partly structural since the model was trained on Chris's labels and evaluated against them, but the gap is large enough to be meaningful
- The high inter-rater MAE confirms the task carries substantial inherent subjectivity
- Results are consistent with expectations for a single-rater first-cut tool